# H2 Analysis Plan — Governance Quality → Sustainability
**Plan file:** `06_H2_plan.md`
**Plan code:** `h2`
**Dataset:** `data/3_processed_data/master_features/panel_dataset_2020_2023.parquet`
**Date:** 2026-05-26

> Tests whether quality of governance (WGI composite) has a statistically significant positive relationship with sustainability outcomes (SDG Index score) across 26 countries, 2020–2023. One-tailed directional hypothesis test built on top of the RQ2 panel regression.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

try:
    from linearmodels.panel import PanelOLS, RandomEffects
    HAS_LINEARMODELS = True
except ImportError:
    HAS_LINEARMODELS = False
    print("⚠️  linearmodels not installed")

# ── Absolute paths ────────────────────────────────────────────────────────────
BASE     = Path("/Users/mohamedinas/Desktop/SE_projects/9_fathima_stat_support/version_v1/8_interview_prep")
DATA     = BASE / "data/3_processed_data/master_features/panel_dataset_2020_2023.parquet"
OUT      = BASE / "outputs"
RQ2_REG  = OUT / "rq2_regression_results.csv"
OUT.mkdir(exist_ok=True)

# ── McKinsey dark-navy style constants ────────────────────────────────────────
BG_DARK    = "#0A1628"
GRID_COLOR = "#1E3A5F"
TEXT_WHITE = "#FFFFFF"
TEXT_DARK  = "#0A1628"
C1         = "#4FC3F7"   # sky blue  — Developed
C2         = "#4DB6AC"   # teal      — Developing
C3         = "#1A73E8"   # blue      — Lower Governance
C4         = "#FFD600"   # gold      — accent
C5         = "#FF7043"   # orange

GROUP_COLORS = {"Developed": C1, "Developing": C2, "Lower Governance": C3}

MCKINSEY_RC = {
    "figure.facecolor": BG_DARK, "figure.figsize": (14, 7), "figure.dpi": 150,
    "axes.facecolor": BG_DARK, "axes.edgecolor": GRID_COLOR,
    "axes.labelcolor": TEXT_WHITE, "axes.titlecolor": TEXT_WHITE,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlepad": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.spines.bottom": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID_COLOR, "grid.linestyle": "--", "grid.linewidth": 0.6,
    "lines.linewidth": 2.5,
    "xtick.color": TEXT_WHITE, "ytick.color": TEXT_WHITE,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.facecolor": BG_DARK, "legend.framealpha": 0.0,
    "legend.labelcolor": TEXT_WHITE, "legend.fontsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "text.color": TEXT_WHITE,
    "savefig.facecolor": BG_DARK, "savefig.bbox": "tight", "savefig.dpi": 150,
}
plt.rcParams.update(MCKINSEY_RC)
print("✅ Setup complete — McKinsey theme loaded")
print(f"   BASE: {BASE}")
print(f"   DATA exists: {DATA.exists()}")

✅ Setup complete — McKinsey theme loaded
   BASE: /Users/mohamedinas/Desktop/SE_projects/9_fathima_stat_support/version_v1/8_interview_prep
   DATA exists: True


## Data Loading
Loading the panel dataset and verifying structure.

In [2]:
prefix = "h2"
x_col  = "wgi_composite"
y_col  = "sdg_index_score"

df = pd.read_parquet(DATA)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Countries: {df['country'].nunique()} | Years: {sorted(df['year'].unique())} | Groups: {df['country_group'].unique().tolist()}")
print("\nColumn dtypes:")
print(df.dtypes.to_string())
df.head(8)

Dataset shape: 104 rows × 7 columns
Countries: 26 | Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] | Groups: ['Developed', 'Developing', 'Lower Governance']

Column dtypes:
iso_code                  str
country                   str
country_group             str
year                    int64
ai_readiness_score    float64
sdg_index_score       float32
wgi_composite         float32


,iso_code,country,country_group,year,ai_readiness_score,sdg_index_score,wgi_composite
0,CAN,Canada,Developed,2020,73.158,79.370003,1.504060
1,CAN,Canada,Developed,2021,77.730,79.040001,1.452090
2,CAN,Canada,Developed,2022,77.390,79.209999,1.430138
3,CAN,Canada,Developed,2023,77.070,79.279999,1.419729
4,DNK,Denmark,Developed,2020,75.618,85.010002,1.825962
5,DNK,Denmark,Developed,2021,76.960,84.910004,1.874868
6,DNK,Denmark,Developed,2022,74.790,85.029999,1.863013
7,DNK,Denmark,Developed,2023,73.910,85.250000,1.849802


## Step 1 — Extract β₁ from RQ2 Panel Regression

**H2:** Quality of governance has a positive relationship with sustainability.

- **H0:** β₁ ≤ 0 — Governance quality has no positive relationship with sustainability
- **H2 (alt):** β₁ > 0 — Governance quality has a statistically significant positive relationship with sustainability

This is a **one-tailed** test (directional). We extract β₁ for `wgi_composite` from the Hausman-selected
panel model in RQ2 (Fixed Effects, two-way, clustered SEs). The Hausman test in RQ2 gave χ² = 17.60, p < 0.0001, preferring Fixed Effects.

In [3]:
from scipy.stats import t as t_dist

# ── Load RQ2 regression results ──────────────────────────────────────────────
rq2_reg = pd.read_csv(RQ2_REG)
print("=== RQ2 Regression Results (source for H2) ===")
display(rq2_reg)

# ── Extract Fixed Effects row (Hausman-selected) ─────────────────────────────
fe_row = rq2_reg[rq2_reg["Model"].str.contains("Fixed Effects")].iloc[0]
beta1  = float(fe_row["beta1 (IV)"])
se     = float(fe_row["SE"])
r2_fe  = float(fe_row["R2"])
n_obs  = int(df[[x_col, y_col]].dropna().shape[0])
df_deg = n_obs - 2

# ── One-tailed test (H2: β₁ > 0) ─────────────────────────────────────────────
t_stat    = beta1 / se
p_onetail = float(t_dist.sf(t_stat, df=df_deg))
ci_lower  = beta1 - 1.645 * se

print(f"\n=== H2 Hypothesis Test (One-Tailed, Fixed Effects model) ===")
print(f"  Selected model : Fixed Effects (Two-Way, clustered SEs) — Hausman χ²=17.60, p<0.0001")
print(f"  β₁ (wgi_composite) = {beta1:.4f}")
print(f"  Standard Error     = {se:.4f}")
print(f"  t-statistic        = {t_stat:.4f}")
print(f"  p-value (one-tail) = {p_onetail:.4f}")
print(f"  95% one-tailed CI lower bound = {ci_lower:.4f}")
print()

if beta1 > 0 and p_onetail < 0.05:
    h2_decision = "✅ REJECT H0 — H2 SUPPORTED (β₁ > 0, p < 0.05 one-tailed)"
else:
    h2_decision = f"❌ FAIL TO REJECT H0 — H2 NOT SUPPORTED at α=0.05 (β₁={beta1:.4f}>0 but p={p_onetail:.4f}>0.05)"

print(h2_decision)
print()
print("Note: FE model strips between-country variance. WGI changes slowly within")
print("a country over 4 years — within-country FE power is inherently limited.")
print("Cross-sectional evidence (reported in Steps 2–3) strongly supports H2's direction.")

=== RQ2 Regression Results (source for H2) ===


,Model,beta1 (IV),SE,t-stat,p-value (2-tail),CI lower,CI upper,R2
0,Pooled OLS,7.9755,0.4601,17.3361,0.0000,7.0738,8.8772,0.7384
1,Fixed Effects (Two-Way),0.9655,0.7512,1.2852,0.2027,-0.5314,2.4623,0.0177
2,Random Effects,2.6981,0.6275,4.2998,0.0000,1.4535,3.9427,0.0960



=== H2 Hypothesis Test (One-Tailed, Fixed Effects model) ===
  Selected model : Fixed Effects (Two-Way, clustered SEs) — Hausman χ²=17.60, p<0.0001
  β₁ (wgi_composite) = 0.9655
  Standard Error     = 0.7512
  t-statistic        = 1.2853
  p-value (one-tail) = 0.1008
  95% one-tailed CI lower bound = -0.2702

❌ FAIL TO REJECT H0 — H2 NOT SUPPORTED at α=0.05 (β₁=0.9655>0 but p=0.1008>0.05)

Note: FE model strips between-country variance. WGI changes slowly within
a country over 4 years — within-country FE power is inherently limited.
Cross-sectional evidence (reported in Steps 2–3) strongly supports H2's direction.


## Step 2 — Effect Size Reporting

Three complementary effect size measures:
1. **Cohen's f²** — based on within-R² from the selected FE model
2. **Standardized β₁** — z-scored pooled OLS for scale-free comparison
3. **Pearson r & Spearman ρ** — bivariate correlations as intuitive benchmarks

In [4]:
from scipy.stats import pearsonr, spearmanr

# ── Cohen's f² ────────────────────────────────────────────────────────────────
cohens_f2    = r2_fe / (1 - r2_fe)
f2_magnitude = "small" if cohens_f2 < 0.15 else ("medium" if cohens_f2 < 0.35 else "large")

# ── Standardized β₁ (pooled OLS) ─────────────────────────────────────────────
valid    = df[[x_col, y_col]].dropna()
x_z      = (valid[x_col] - valid[x_col].mean()) / valid[x_col].std()
y_z      = (valid[y_col] - valid[y_col].mean()) / valid[y_col].std()
std_beta = float(sm.OLS(y_z, sm.add_constant(x_z)).fit().params[x_col])

# ── Pooled correlations ───────────────────────────────────────────────────────
r_p, p_p = pearsonr(valid[x_col], valid[y_col])
r_s, p_s = spearmanr(valid[x_col], valid[y_col])
n_valid  = len(valid)
z_r      = np.arctanh(r_p)
se_z     = 1 / np.sqrt(n_valid - 3)
r_ci_l   = np.tanh(z_r - 1.96 * se_z)
r_ci_u   = np.tanh(z_r + 1.96 * se_z)

effect_table = pd.DataFrame([
    {"Measure": "Cohen's f² (FE within-R²=0.0177)",
     "Value": round(cohens_f2,4), "Magnitude": f"{f2_magnitude} effect",
     "Note": "FE strips between-country variance → low within-R²"},
    {"Measure": "Standardized β₁ (Pooled OLS)",
     "Value": round(std_beta,4), "Magnitude": "large",
     "Note": "Equals Pearson r — strong cross-sectional association"},
    {"Measure": f"Pearson r (pooled, n={n_valid})",
     "Value": round(r_p,4), "Magnitude": f"95% CI [{r_ci_l:.3f}, {r_ci_u:.3f}]",
     "Note": f"p = {p_p:.2e}"},
    {"Measure": f"Spearman ρ (pooled, n={n_valid})",
     "Value": round(r_s,4), "Magnitude": f"p = {p_s:.2e}",
     "Note": "Rank-based; robust to non-linearity"},
])

print("=== Effect Sizes ===")
display(effect_table)
print(f"\nCohen's f² = {cohens_f2:.4f} ({f2_magnitude} — within-FE estimate)")
print(f"Std β₁     = {std_beta:.4f} (large cross-sectional pooled effect)")
print(f"Pearson r  = {r_p:.4f}  (p = {p_p:.2e})")
print(f"Spearman ρ = {r_s:.4f}  (p = {p_s:.2e})")

=== Effect Sizes ===


,Measure,Value,Magnitude,Note
0,Cohen's f² (FE within-R²=0.0177),0.0180,small effect,FE strips between-country variance → low withi...
1,Standardized β₁ (Pooled OLS),0.8593,large,Equals Pearson r — strong cross-sectional asso...
2,"Pearson r (pooled, n=104)",0.8593,"95% CI [0.799, 0.903]",p = 1.83e-31
3,"Spearman ρ (pooled, n=104)",0.8078,p = 3.77e-25,Rank-based; robust to non-linearity



Cohen's f² = 0.0180 (small — within-FE estimate)
Std β₁     = 0.8593 (large cross-sectional pooled effect)
Pearson r  = 0.8593  (p = 1.83e-31)
Spearman ρ = 0.8078  (p = 3.77e-25)


## Step 3 — Additional Evidence: Between-Group Analysis

Substantiates H2 using the three-tier governance stratification:
- *Developed* (highest WGI) → *Developing* (middle) → *Lower Governance* (lowest)

**Expectation:** mean SDG decreases monotonically as governance tier decreases.

In [5]:
group_order  = ["Developed", "Developing", "Lower Governance"]
group_stats  = (df.groupby("country_group")[y_col]
                .agg(["mean","std","count","sem"])
                .round(3)
                .reindex(group_order))
group_stats.columns = ["Mean SDG", "Std Dev", "N obs", "SE"]

print("=== Mean SDG Index Score by Governance Tier ===")
display(group_stats)

means    = group_stats["Mean SDG"].values
mono_ok  = means[0] > means[1] > means[2]
print(f"\nMonotonically decreasing gradient: {'✅ YES — consistent with H2' if mono_ok else '❌ NO'}")
print(f"  Developed        → {means[0]:.2f}")
print(f"  Developing       → {means[1]:.2f}")
print(f"  Lower Governance → {means[2]:.2f}")
print(f"  Gap (Developed − Lower Gov): {means[0] - means[2]:.2f} SDG points")

group_stats.to_csv(OUT / f"{prefix}_group_comparison.csv")
print(f"\n✅ Saved: outputs/{prefix}_group_comparison.csv")

=== Mean SDG Index Score by Governance Tier ===


,Mean SDG,Std Dev,N obs,SE
country_group,,,,
Developed,80.745003,5.169,40,0.817
Developing,70.031998,3.442,40,0.544
Lower Governance,58.703999,4.052,24,0.827



Monotonically decreasing gradient: ✅ YES — consistent with H2
  Developed        → 80.75
  Developing       → 70.03
  Lower Governance → 58.70
  Gap (Developed − Lower Gov): 22.04 SDG points

✅ Saved: outputs/h2_group_comparison.csv


In [6]:
from scipy.stats import shapiro, f_oneway, kruskal, mannwhitneyu
from itertools import combinations

groups_data = {g: df.loc[df["country_group"]==g, y_col].dropna().values
               for g in group_order}

print("=== Shapiro-Wilk Normality Tests ===")
all_normal = True
for g, data in groups_data.items():
    stat_sw, p_sw = shapiro(data)
    norm_flag = "✅ Normal" if p_sw >= 0.05 else "⚠️  Non-normal"
    print(f"  {g:<22}: W = {stat_sw:.4f}, p = {p_sw:.4f}  {norm_flag}")
    if p_sw < 0.05:
        all_normal = False

print(f"\nUsing: {'One-way ANOVA' if all_normal else 'Kruskal-Wallis (non-parametric)'}")

if all_normal:
    stat_kw, p_kw = f_oneway(*groups_data.values())
    test_name = "ANOVA F"
else:
    stat_kw, p_kw = kruskal(*groups_data.values())
    test_name = "Kruskal-Wallis H"

print(f"\n{test_name}-statistic = {stat_kw:.4f},  p = {p_kw:.4e}")

if p_kw < 0.05:
    print("\n✅ Significant group differences → post-hoc pairwise tests (Mann-Whitney U + Bonferroni)")
    bonf_alpha = 0.05 / 3
    pair_rows  = []
    for g1, g2 in combinations(group_order, 2):
        u_stat, p_mw = mannwhitneyu(groups_data[g1], groups_data[g2], alternative="two-sided")
        sig_flag = "✅ Sig." if p_mw < bonf_alpha else "—"
        print(f"   {g1} vs {g2}: U={u_stat:.1f}, p={p_mw:.4f}  {sig_flag}")
        pair_rows.append({"Pair": f"{g1} vs {g2}", "U": round(u_stat,1),
                          "p-value": round(p_mw,4), "Significant (Bonferroni α=0.017)": p_mw < bonf_alpha})
    display(pd.DataFrame(pair_rows))
else:
    print("No significant group differences detected (p ≥ 0.05)")

=== Shapiro-Wilk Normality Tests ===
  Developed             : W = 0.8928, p = 0.0012  ⚠️  Non-normal
  Developing            : W = 0.9366, p = 0.0265  ⚠️  Non-normal
  Lower Governance      : W = 0.8289, p = 0.0009  ⚠️  Non-normal

Using: Kruskal-Wallis (non-parametric)

Kruskal-Wallis H-statistic = 82.2076,  p = 1.4088e-18

✅ Significant group differences → post-hoc pairwise tests (Mann-Whitney U + Bonferroni)
   Developed vs Developing: U=1506.5, p=0.0000  ✅ Sig.
   Developed vs Lower Governance: U=960.0, p=0.0000  ✅ Sig.
   Developing vs Lower Governance: U=959.0, p=0.0000  ✅ Sig.


,Pair,U,p-value,Significant (Bonferroni α=0.017)
0,Developed vs Developing,1506.5,0.0,True
1,Developed vs Lower Governance,960.0,0.0,True
2,Developing vs Lower Governance,959.0,0.0,True


In [7]:
country_means = df.groupby("country")[[x_col, y_col]].mean().round(3)
country_means["wgi_rank"] = country_means[x_col].rank(ascending=True)
country_means["sdg_rank"] = country_means[y_col].rank(ascending=True)

r_spear, p_spear = spearmanr(country_means[x_col], country_means[y_col])
n_ctry  = len(country_means)
z_sp    = np.arctanh(r_spear)
se_sp   = 1 / np.sqrt(n_ctry - 3)
sp_ci_l = np.tanh(z_sp - 1.96 * se_sp)
sp_ci_u = np.tanh(z_sp + 1.96 * se_sp)

print(f"=== Country-Level Spearman Rank Correlation (N = {n_ctry} countries) ===")
print(f"  Spearman ρ = {r_spear:.4f}")
print(f"  p-value    = {p_spear:.4e}")
print(f"  95% CI     = [{sp_ci_l:.4f}, {sp_ci_u:.4f}]")
print()
if r_spear > 0 and p_spear < 0.05:
    print("✅ Significant positive rank correlation — higher governance rank → higher SDG rank")
    print("   Strong country-level structural evidence consistent with H2")

print(f"\n=== Country Rankings (sorted by governance quality) ===")
display(country_means[[x_col, "wgi_rank", y_col, "sdg_rank"]]
        .sort_values("wgi_rank", ascending=False).round(3))

=== Country-Level Spearman Rank Correlation (N = 26 countries) ===
  Spearman ρ = 0.8140
  p-value    = 4.2257e-07
  95% CI     = [0.6231, 0.9134]

✅ Significant positive rank correlation — higher governance rank → higher SDG rank
   Strong country-level structural evidence consistent with H2

=== Country Rankings (sorted by governance quality) ===


,wgi_composite,wgi_rank,sdg_index_score,sdg_rank
country,,,,
Finland,1.854,26.0,86.931999,26.0
Denmark,1.853,25.0,85.050003,24.0
Norway,1.765,24.0,82.580002,22.0
Sweden,1.675,23.0,85.888000,25.0
Netherlands,1.626,22.0,79.805000,20.0
Singapore,1.530,21.0,69.297997,11.0
Germany,1.480,20.0,83.434998,23.0
Canada,1.452,19.0,79.224998,19.0
Japan,1.417,18.0,80.112000,21.0


## Step 4 — Robustness Checks for H2

1. **Cook's Distance** — identify influential country-years; re-run without them
2. **Year-by-year cross-sections** — check if governance coefficient is consistently positive
3. **Individual WGI dimensions** — data availability note
4. **Alternative WGI scaling** — raw (−2.5 to +2.5) vs percentile rank (0–100)

In [8]:
# ── Cook's Distance ───────────────────────────────────────────────────────────
valid_r   = df[[x_col, y_col, "country", "year"]].dropna().copy()
X_rob     = sm.add_constant(valid_r[x_col])
rob_res   = sm.OLS(valid_r[y_col], X_rob).fit()
influence = rob_res.get_influence()
cooks_d   = influence.cooks_distance[0]
threshold = 4 / len(valid_r)
n_outliers = (cooks_d > threshold).sum()

print(f"Cook's D threshold (4/n = 4/{len(valid_r)}): {threshold:.4f}")
print(f"Influential observations (Cook's D > threshold): {n_outliers}")

outlier_idx = np.where(cooks_d > threshold)[0]
if len(outlier_idx) > 0:
    print("\nInfluential observations:")
    for idx in outlier_idx[:12]:
        row = valid_r.iloc[idx]
        print(f"  {row['country']} {row['year']}: Cook's D = {cooks_d[idx]:.4f}")
    valid_clean = valid_r.drop(valid_r.index[outlier_idx]).copy()
    X_clean     = sm.add_constant(valid_clean[x_col])
    res_clean   = sm.OLS(valid_clean[y_col], X_clean).fit(cov_type="HC3")
    print(f"\nModel without outliers: β₁ = {res_clean.params[x_col]:.4f}, p = {res_clean.pvalues[x_col]:.4f}")
    print(f"Full model:              β₁ = {rob_res.params[x_col]:.4f}, p = {rob_res.pvalues[x_col]:.4f}")
else:
    print("No influential observations detected.")

# ── Year-by-year OLS ──────────────────────────────────────────────────────────
print("\n=== Year-by-Year Cross-Sectional OLS ===")
yr_results = []
for yr in sorted(df["year"].unique()):
    sub = df.loc[df["year"]==yr, [x_col, y_col]].dropna()
    if len(sub) < 4: continue
    X_yr = sm.add_constant(sub[x_col])
    ry   = sm.OLS(sub[y_col], X_yr).fit()
    yr_results.append({
        "Year": yr, "N": len(sub),
        "β₁": round(float(ry.params[x_col]), 4),
        "SE":  round(float(ry.bse[x_col]), 4),
        "p-value": round(float(ry.pvalues[x_col]), 4),
        "R²":  round(ry.rsquared, 4),
        "Direction": "+" if ry.params[x_col] > 0 else "−"
    })
yr_df = pd.DataFrame(yr_results)
display(yr_df)
all_positive = all(yr_df["β₁"] > 0)
print(f"\nAll year-by-year β₁ positive: {'✅ YES — sign consistent across all 4 years' if all_positive else '⚠️  Mixed'}")

yr_df.to_csv(OUT / f"{prefix}_robustness_checks.csv", index=False)
print(f"✅ Saved: outputs/{prefix}_robustness_checks.csv")

Cook's D threshold (4/n = 4/104): 0.0385
Influential observations (Cook's D > threshold): 5

Influential observations:
  Singapore 2020: Cook's D = 0.0753
  Singapore 2021: Cook's D = 0.0762
  Singapore 2022: Cook's D = 0.0626
  Singapore 2023: Cook's D = 0.0546
  Ethiopia 2020: Cook's D = 0.0390

Model without outliers: β₁ = 8.4343, p = 0.0000
Full model:              β₁ = 7.9755, p = 0.0000

=== Year-by-Year Cross-Sectional OLS ===


,Year,N,β₁,SE,p-value,R²,Direction
0,2020,26,8.1679,0.9845,0.0,0.7415,+
1,2021,26,7.8947,0.9476,0.0,0.7431,+
2,2022,26,7.8505,0.9644,0.0,0.7341,+
3,2023,26,8.0322,0.9643,0.0,0.7430,+



All year-by-year β₁ positive: ✅ YES — sign consistent across all 4 years
✅ Saved: outputs/h2_robustness_checks.csv


In [9]:
# ── Individual WGI dimensions ────────────────────────────────────────────────
print("ℹ️  Individual WGI dimension analysis (VA, PV, GE, RQ, RL, CC):")
print("   Sub-dimensions are NOT present in panel_dataset_2020_2023.parquet.")
print("   Only wgi_composite (equal-weight average of all 6 dimensions) is available.")
print("   Dimension-level analysis requires re-joining from World Bank WGI raw data.")
print("   → Skipped; composite score is used throughout.\n")

# ── Alternative WGI scaling: percentile rank ──────────────────────────────────
df_alt = df.copy()
df_alt["wgi_pct_rank"] = df_alt["wgi_composite"].rank(pct=True) * 100

res_raw = sm.OLS(df_alt[y_col], sm.add_constant(df_alt[x_col])).fit(cov_type="HC3")
res_pct = sm.OLS(df_alt[y_col], sm.add_constant(df_alt["wgi_pct_rank"])).fit(cov_type="HC3")

alt_df = pd.DataFrame([
    {"Scaling": "Raw WGI (−2.5 to +2.5)",
     "β₁": round(float(res_raw.params[x_col]),4),
     "SE":  round(float(res_raw.bse[x_col]),4),
     "p-value": round(float(res_raw.pvalues[x_col]),4),
     "R²":  round(res_raw.rsquared,4)},
    {"Scaling": "Percentile Rank (0–100)",
     "β₁": round(float(res_pct.params["wgi_pct_rank"]),4),
     "SE":  round(float(res_pct.bse["wgi_pct_rank"]),4),
     "p-value": round(float(res_pct.pvalues["wgi_pct_rank"]),4),
     "R²":  round(res_pct.rsquared,4)},
])
print("=== Alternative WGI Scaling (Pooled OLS) ===")
display(alt_df)

both_sig = (res_raw.pvalues[x_col] < 0.05) and (res_pct.pvalues["wgi_pct_rank"] < 0.05)
print(f"\nSignificant under both scalings: {'✅ YES — scale-independent' if both_sig else '⚠️  Scale-dependent'}")

alt_df.to_csv(OUT / f"{prefix}_dimension_analysis.csv", index=False)
print(f"✅ Saved: outputs/{prefix}_dimension_analysis.csv")

ℹ️  Individual WGI dimension analysis (VA, PV, GE, RQ, RL, CC):
   Sub-dimensions are NOT present in panel_dataset_2020_2023.parquet.
   Only wgi_composite (equal-weight average of all 6 dimensions) is available.
   Dimension-level analysis requires re-joining from World Bank WGI raw data.
   → Skipped; composite score is used throughout.

=== Alternative WGI Scaling (Pooled OLS) ===


,Scaling,β₁,SE,p-value,R²
0,Raw WGI (−2.5 to +2.5),7.9755,0.4601,0.0,0.7384
1,Percentile Rank (0–100),0.2728,0.0151,0.0,0.6923



Significant under both scalings: ✅ YES — scale-independent
✅ Saved: outputs/h2_dimension_analysis.csv


## Step 5 — Visualizations for H2

Four McKinsey dark-navy charts:
1. Scatter — WGI vs SDG, color-coded by country group with OLS trend
2. Bar chart — Mean SDG by governance tier (group gradient)
3. Coefficient plot — β₁ across panel models and each year
4. Heatmap — WGI composite + SDG side-by-side (countries × years)

In [10]:
fig, ax = plt.subplots()
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_DARK)
for s in ax.spines.values(): s.set_visible(False)

for group, color in GROUP_COLORS.items():
    mask = df["country_group"] == group
    ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
               color=color, s=65, alpha=0.85, zorder=5, label=group)

# Country labels
ctry_means = df.groupby("country")[[x_col, y_col]].mean()
for ctry, row in ctry_means.iterrows():
    ax.text(row[x_col] + 0.03, row[y_col], ctry[:3].upper(),
            color=TEXT_WHITE, fontsize=6.2, va="center", alpha=0.8)

# OLS trend + CI band
x_vals   = df[x_col].dropna()
y_vals   = df.loc[x_vals.index, y_col].dropna()
com_idx  = x_vals.index.intersection(y_vals.index)
x_v, y_v = x_vals[com_idx].values, y_vals[com_idx].values
m, b_    = np.polyfit(x_v, y_v, 1)
x_line   = np.linspace(x_v.min(), x_v.max(), 100)
ax.plot(x_line, m*x_line+b_, color=TEXT_WHITE, lw=1.5, ls="--", alpha=0.7,
        label=f"OLS trend (slope={m:.2f})")
from scipy import stats as _st
n_sc = len(x_v); xm_sc = x_v.mean()
se_sc = (np.sqrt(sum((y_v-(m*x_v+b_))**2)/(n_sc-2))
         * np.sqrt(1/n_sc + (x_line-xm_sc)**2/sum((x_v-xm_sc)**2)))
t_c = _st.t.ppf(0.975, n_sc-2)
ax.fill_between(x_line, m*x_line+b_-t_c*se_sc, m*x_line+b_+t_c*se_sc,
                color=TEXT_WHITE, alpha=0.08)

ax.set_xlabel("WGI Composite (Governance Quality)", color=TEXT_WHITE, fontsize=10)
ax.set_ylabel("SDG Index Score (Sustainability)", color=TEXT_WHITE, fontsize=10)
ax.set_title("WGI Composite vs SDG Index Score\n(n=104 country-years  |  colour = country group  |  dashed = OLS trend ± 95% CI)",
             fontweight="bold", color=TEXT_WHITE)
ax.legend(loc="upper left", framealpha=0.0)
plt.tight_layout()
plt.savefig(OUT / f"{prefix}_scatter_plot.png", bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"✅ Saved: outputs/{prefix}_scatter_plot.png")

✅ Saved: outputs/h2_scatter_plot.png


In [11]:
group_bar = (df.groupby("country_group")[y_col]
             .agg(["mean","sem"]).reset_index())
group_bar.columns = ["group","mean","sem"]
group_bar = (group_bar.set_index("group")
             .reindex(["Developed","Developing","Lower Governance"])
             .reset_index())

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_DARK)
for s in ax.spines.values(): s.set_visible(False)

colors_bar = [GROUP_COLORS[g] for g in group_bar["group"]]
bars = ax.bar(group_bar["group"], group_bar["mean"],
              color=colors_bar, width=0.55, zorder=5, alpha=0.9)
ax.errorbar(group_bar["group"], group_bar["mean"], yerr=group_bar["sem"],
            fmt="none", color=TEXT_WHITE, capsize=7, lw=1.8, zorder=6)

for bar, val in zip(bars, group_bar["mean"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.7,
            f"{val:.1f}", ha="center", va="bottom", color=TEXT_WHITE,
            fontweight="bold", fontsize=12)

ax.set_ylabel("Mean SDG Index Score", color=TEXT_WHITE, fontsize=10)
ax.set_title("Mean SDG Index Score by Governance Tier\n(error bars = ±1 SE  |  monotonic gradient supports H2)",
             fontweight="bold", color=TEXT_WHITE)
ax.set_ylim(0, group_bar["mean"].max() * 1.18)
ax.tick_params(axis="x", labelsize=11, colors=TEXT_WHITE)
plt.tight_layout()
plt.savefig(OUT / f"{prefix}_group_bar_chart.png", bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"✅ Saved: outputs/{prefix}_group_bar_chart.png")

✅ Saved: outputs/h2_group_bar_chart.png

In [12]:
coef_rows = []
model_labels = {
    "Pooled OLS":              "Pooled OLS",
    "Fixed Effects (Two-Way)": "Fixed Effects (FE) ← Selected",
    "Random Effects":          "Random Effects"
}
for _, row in rq2_reg.iterrows():
    lbl = model_labels.get(row["Model"], row["Model"])
    coef_rows.append({
        "Model": lbl, "β₁": float(row["beta1 (IV)"]),
        "CI lower": float(row["CI lower"]), "CI upper": float(row["CI upper"]),
        "p-value": float(row["p-value (2-tail)"]), "highlight": "Selected" in lbl
    })
for _, row in yr_df.iterrows():
    b_yr = float(row["β₁"]); se_yr = float(row["SE"])
    coef_rows.append({
        "Model": f"OLS {int(row['Year'])} (n={int(row['N'])})", "β₁": b_yr,
        "CI lower": b_yr - 1.96*se_yr, "CI upper": b_yr + 1.96*se_yr,
        "p-value": float(row["p-value"]), "highlight": False
    })
coef_df = pd.DataFrame(coef_rows)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_DARK)
for s in ax.spines.values(): s.set_visible(False)
ax.axvline(0, color=TEXT_WHITE, lw=1, ls="-", alpha=0.35, zorder=3)

for i, row in coef_df.iterrows():
    color = C4 if row["highlight"] else (C2 if "OLS 2" in row["Model"] else C1)
    ax.plot([row["CI lower"], row["CI upper"]], [i, i],
            color=color, lw=3, solid_capstyle="round", zorder=4, alpha=0.85)
    ax.scatter(row["β₁"], i, color=color, s=90, zorder=5)
    x_txt = max(row["CI upper"], row["β₁"]) + 0.15
    ax.text(x_txt, i, f"β={row['β₁']:.3f}  p={row['p-value']:.3f}",
            color=TEXT_WHITE, fontsize=7.5, va="center")

ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df["Model"], color=TEXT_WHITE, fontsize=8)
ax.set_xlabel("Coefficient β₁ (WGI Composite)", color=TEXT_WHITE, fontsize=10)
ax.set_title("Coefficient Plot — β₁ (WGI Composite) Across Models\n(line = 95% CI  |  dot = point estimate  |  gold = selected model)",
             fontweight="bold", color=TEXT_WHITE)
plt.tight_layout()
plt.savefig(OUT / f"{prefix}_coefficient_plot.png", bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"✅ Saved: outputs/{prefix}_coefficient_plot.png")

✅ Saved: outputs/h2_coefficient_plot.png


In [13]:
import matplotlib.colors as mcolors

pivot_wgi = df.pivot_table(index="country", columns="year", values=x_col, aggfunc="mean")
pivot_sdg = df.pivot_table(index="country", columns="year", values=y_col, aggfunc="mean")
ctry_order = pivot_wgi.mean(axis=1).sort_values(ascending=False).index
pivot_wgi  = pivot_wgi.reindex(ctry_order)
pivot_sdg  = pivot_sdg.reindex(ctry_order)

cmap_wgi = mcolors.LinearSegmentedColormap.from_list("wgi_cmap", [BG_DARK, C1])
cmap_sdg = mcolors.LinearSegmentedColormap.from_list("sdg_cmap", [BG_DARK, C2])

fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.patch.set_facecolor(BG_DARK)

for ax, pivot, cmap, tlabel in zip(
        axes, [pivot_wgi, pivot_sdg], [cmap_wgi, cmap_sdg],
        ["WGI Composite\n(Governance Quality)", "SDG Index Score\n(Sustainability)"]):
    ax.set_facecolor(BG_DARK)
    for s in ax.spines.values(): s.set_visible(False)
    im  = ax.imshow(pivot.values, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, color=TEXT_WHITE, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, color=TEXT_WHITE, fontsize=8)
    valid_vals = pivot.values[~np.isnan(pivot.values)]
    v_max = valid_vals.max()
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                tc = TEXT_DARK if val > v_max * 0.6 else TEXT_WHITE
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        color=tc, fontsize=6.5, fontweight="bold")
    cbar = plt.colorbar(im, ax=ax, shrink=0.55, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=TEXT_WHITE)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_WHITE, fontsize=8)
    ax.set_title(f"{tlabel}\n(sorted highest → lowest WGI)",
                 fontweight="bold", color=TEXT_WHITE, pad=12)

plt.suptitle("Side-by-Side Heatmaps: Countries × Years (2020–2023)\nVisual alignment of governance quality and sustainability outcomes supports H2",
             color=TEXT_WHITE, fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUT / f"{prefix}_heatmap.png", bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"✅ Saved: outputs/{prefix}_heatmap.png")

✅ Saved: outputs/h2_heatmap.png


## Step 6 — Reporting H2 Results

Compile comprehensive hypothesis test table and save for thesis use.

In [14]:
hyp_rows = []
for _, row in rq2_reg.iterrows():
    b1   = float(row["beta1 (IV)"])
    se_m = float(row["SE"])
    t_m  = b1 / se_m
    p1   = float(t_dist.sf(t_m, df=n_obs-2))
    cl   = b1 - 1.645 * se_m
    cu   = float(row["CI upper"])
    r2_m = float(row["R2"])
    dec  = "Reject H0 ✅" if (b1 > 0 and p1 < 0.05) else "Fail to Reject H0 ❌"
    hyp_rows.append({
        "Model": row["Model"],
        "β₁ (WGI Composite)": round(b1,4), "SE": round(se_m,4),
        "t-stat (1-tail)": round(t_m,4), "p-value (1-tail)": round(p1,4),
        "95% CI lower (1-tail)": round(cl,4), "95% CI upper": round(cu,4),
        "R²": round(r2_m,4), "Decision": dec
    })
# Selected model row
fe_row2 = rq2_reg[rq2_reg["Model"]=="Fixed Effects (Two-Way)"].iloc[0]
fe_b1 = float(fe_row2["beta1 (IV)"])
fe_se = float(fe_row2["SE"])
fe_t  = fe_b1 / fe_se
fe_p1 = float(t_dist.sf(fe_t, df=n_obs-2))
hyp_rows.append({
    "Model": "★ Selected (FE, Hausman χ²=17.60 p<0.0001)",
    "β₁ (WGI Composite)": round(fe_b1,4), "SE": round(fe_se,4),
    "t-stat (1-tail)": round(fe_t,4), "p-value (1-tail)": round(fe_p1,4),
    "95% CI lower (1-tail)": round(fe_b1 - 1.645*fe_se,4),
    "95% CI upper": round(float(fe_row2["CI upper"]),4),
    "R²": round(float(fe_row2["R2"]),4),
    "Decision": f"Fail to Reject H0 ❌ (β₁>0 but p={fe_p1:.4f}>0.05)"
})

hyp_df = pd.DataFrame(hyp_rows)
print("=== H2 Full Hypothesis Test Results Table ===")
display(hyp_df)

hyp_df.to_csv(OUT / f"{prefix}_hypothesis_test_table.csv", index=False)
print(f"\n✅ Saved: outputs/{prefix}_hypothesis_test_table.csv")
print(f"\nSummary:")
print(f"  FE β₁ = {fe_b1:.4f}  (SE={fe_se:.4f}, t={fe_t:.4f}, p one-tail={fe_p1:.4f})")
print(f"  Group gradient: Developed(80.7) > Developing(70.0) > LowerGov(58.7)  ✅")
print(f"  Kruskal-Wallis H=82.2, p<0.0001  ✅")
print(f"  Country-level Spearman ρ={r_spear:.4f}, p<0.0001  ✅")
print(f"  Pooled Pearson r={r_p:.4f}, p<0.001  ✅")
print(f"  Conclusion: H2 not formally supported at α=0.05 (FE within-country estimate).")
print(f"  Cross-sectional evidence overwhelmingly supports governance–sustainability link.")

=== H2 Full Hypothesis Test Results Table ===


,Model,β₁ (WGI Composite),SE,t-stat (1-tail),p-value (1-tail),95% CI lower (1-tail),95% CI upper,R²,Decision
0,Pooled OLS,7.9755,0.4601,17.3343,0.0000,7.2186,8.8772,0.7384,Reject H0 ✅
1,Fixed Effects (Two-Way),0.9655,0.7512,1.2853,0.1008,-0.2702,2.4623,0.0177,Fail to Reject H0 ❌
2,Random Effects,2.6981,0.6275,4.2998,0.0000,1.6659,3.9427,0.0960,Reject H0 ✅
3,"★ Selected (FE, Hausman χ²=17.60 p<0.0001)",0.9655,0.7512,1.2853,0.1008,-0.2702,2.4623,0.0177,Fail to Reject H0 ❌ (β₁>0 but p=0.1008>0.05)



✅ Saved: outputs/h2_hypothesis_test_table.csv

Summary:
  FE β₁ = 0.9655  (SE=0.7512, t=1.2853, p one-tail=0.1008)
  Group gradient: Developed(80.7) > Developing(70.0) > LowerGov(58.7)  ✅
  Kruskal-Wallis H=82.2, p<0.0001  ✅
  Country-level Spearman ρ=0.8140, p<0.0001  ✅
  Pooled Pearson r=0.8593, p<0.001  ✅
  Conclusion: H2 not formally supported at α=0.05 (FE within-country estimate).
  Cross-sectional evidence overwhelmingly supports governance–sustainability link.


## 📊 Plain-Language Synthesis
> *Written for readers who are not statisticians*

---

### What did we study?

We examined whether countries with better governance quality — measured by the World Bank's Worldwide Governance Indicators (WGI) composite score — also achieve higher sustainability outcomes, as measured by the SDG Index score. Using a panel of 26 countries across four years (2020–2023), we applied Fixed Effects panel regression (the most rigorous method for this type of data, preferred by the Hausman test: χ² = 17.60, p < 0.0001) to ask: *after accounting for everything permanently different between countries, does a change in governance quality within a country predict a corresponding change in its sustainability score?*

---

### What did we find?

#### Finding 1 — The governance–sustainability gradient is striking across countries
Across all 26 countries, better-governed countries consistently score higher on sustainability. The pooled Pearson correlation is r = 0.86 (p < 0.001) — governance quality explains roughly 74% of the variation in SDG scores. At the country level, the Spearman rank correlation is ρ = 0.81 (p < 0.001, 95% CI [0.63, 0.91]): rank a country by governance quality and you nearly rank it by sustainability performance.

#### Finding 2 — The three governance-tier gradient is perfectly ordered
Countries in the *Developed* governance tier averaged an SDG score of **80.7**, compared to **70.0** in the *Developing* tier and **58.7** in the *Lower Governance* tier — a 22-point spread. This monotonically decreasing gradient is exactly what H2 predicts. A Kruskal-Wallis test confirms these differences are not due to chance (H = 82.2, p < 0.0001), and all three pairwise comparisons are significant after Bonferroni correction.

#### Finding 3 — The formal Fixed Effects test is directionally correct but marginally non-significant
The Fixed Effects regression coefficient is β₁ = 0.9655 (SE = 0.7512, t = 1.2853, p = 0.1008 one-tailed). The direction is consistent with H2 (β₁ > 0), but the result just misses the α = 0.05 threshold. The key explanation: WGI scores change very little within a single country over four years — the FE estimator, which is designed to detect within-country year-to-year changes, has limited power in a short four-year panel. Year-by-year cross-sectional regressions confirm the coefficient is positive in all four years (2020–2023).

---

### How confident are we?

| What we measured | Result | Confidence |
|-----------------|--------|------------|
| FE within-country β₁ (wgi_composite) | 0.9655 (p = 0.10, one-tail) | Marginal — direction correct, borderline significance |
| Cross-sectional Pearson r | 0.86 (p < 0.001) | Very strong |
| Country-level Spearman ρ | 0.81 (p < 0.001) | Very strong |
| Group gradient (K-W test) | H = 82.2, p < 0.0001 | Extremely strong |
| Effect size (Cohen's f², within-FE) | 0.018 (small) | Small within-country |
| Alternative WGI scaling robustness | Significant under both raw & percentile rank | Scale-independent |
| Year-by-year consistency | β₁ positive in all 4 years | Directionally robust |

> **Statistical significance** means the finding is unlikely to be due to chance alone (p < 0.05).
> **Effect size** tells us whether the finding is practically meaningful, not just statistically detectable.

---

### What are the limitations?

- **Short panel / within-country FE power:** The FE model strips out all between-country differences. Because WGI changes slowly within a single country over a 4-year window, the estimator has limited statistical power. A 10–15 year panel would likely yield a significant FE estimate.
- **Observational data:** Governance quality is *associated* with sustainability, but we cannot prove causation. Both could be driven by national wealth or historical institutional development.
- **Sample size:** With 104 country-year observations (26 countries × 4 years), the FE estimate sits at p = 0.10 — marginally outside conventional significance. A slightly larger or longer sample could tip this result.
- **Composite governance measure:** WGI is an average of six sub-dimensions. Individual dimensions (voice & accountability, rule of law, etc.) may have heterogeneous relationships with sustainability that are masked in the composite.

---

### In plain English: Countries with stronger governance consistently achieve higher sustainability scores — the pattern is clear and robust in every cross-sectional test, but the most conservative within-country analysis (Fixed Effects over just four years) falls just short of the 5% significance threshold, meaning we cannot formally rule out chance for within-country year-to-year governance changes in this short window.


In [15]:
synthesis_text = """## 📊 Plain-Language Synthesis
> *Written for readers who are not statisticians*

---

### What did we study?

We examined whether countries with better governance quality — measured by the World Bank's Worldwide Governance Indicators (WGI) composite score — also achieve higher sustainability outcomes, as measured by the SDG Index score. Using a panel of 26 countries across four years (2020–2023), we applied Fixed Effects panel regression (the most rigorous method for this type of data, preferred by the Hausman test: χ² = 17.60, p < 0.0001) to ask: *after accounting for everything permanently different between countries, does a change in governance quality within a country predict a corresponding change in its sustainability score?*

---

### What did we find?

#### Finding 1 — The governance–sustainability gradient is striking across countries
Across all 26 countries, better-governed countries consistently score higher on sustainability. The pooled Pearson correlation is r = 0.86 (p < 0.001) — governance quality explains roughly 74% of the variation in SDG scores. At the country level, the Spearman rank correlation is ρ = 0.81 (p < 0.001, 95% CI [0.63, 0.91]): rank a country by governance quality and you nearly rank it by sustainability performance.

#### Finding 2 — The three governance-tier gradient is perfectly ordered
Countries in the *Developed* governance tier averaged an SDG score of **80.7**, compared to **70.0** in the *Developing* tier and **58.7** in the *Lower Governance* tier — a 22-point spread. This monotonically decreasing gradient is exactly what H2 predicts. A Kruskal-Wallis test confirms these differences are not due to chance (H = 82.2, p < 0.0001), and all three pairwise comparisons are significant after Bonferroni correction.

#### Finding 3 — The formal Fixed Effects test is directionally correct but marginally non-significant
The Fixed Effects regression coefficient is β₁ = 0.9655 (SE = 0.7512, t = 1.2853, p = 0.1008 one-tailed). The direction is consistent with H2 (β₁ > 0), but the result just misses the α = 0.05 threshold. The key explanation: WGI scores change very little within a single country over four years — the FE estimator, which is designed to detect within-country year-to-year changes, has limited power in a short four-year panel. Year-by-year cross-sectional regressions confirm the coefficient is positive in all four years (2020–2023).

---

### How confident are we?

| What we measured | Result | Confidence |
|-----------------|--------|------------|
| FE within-country β₁ (wgi_composite) | 0.9655 (p = 0.10, one-tail) | Marginal — direction correct, borderline significance |
| Cross-sectional Pearson r | 0.86 (p < 0.001) | Very strong |
| Country-level Spearman ρ | 0.81 (p < 0.001) | Very strong |
| Group gradient (K-W test) | H = 82.2, p < 0.0001 | Extremely strong |
| Effect size (Cohen's f², within-FE) | 0.018 (small) | Small within-country |
| Alternative WGI scaling robustness | Significant under both raw & percentile rank | Scale-independent |
| Year-by-year consistency | β₁ positive in all 4 years | Directionally robust |

> **Statistical significance** means the finding is unlikely to be due to chance alone (p < 0.05).
> **Effect size** tells us whether the finding is practically meaningful, not just statistically detectable.

---

### What are the limitations?

- **Short panel / within-country FE power:** The FE model strips out all between-country differences. Because WGI changes slowly within a single country over a 4-year window, the estimator has limited statistical power. A 10–15 year panel would likely yield a significant FE estimate.
- **Observational data:** Governance quality is *associated* with sustainability, but we cannot prove causation. Both could be driven by national wealth or historical institutional development.
- **Sample size:** With 104 country-year observations (26 countries × 4 years), the FE estimate sits at p = 0.10 — marginally outside conventional significance. A slightly larger or longer sample could tip this result.
- **Composite governance measure:** WGI is an average of six sub-dimensions. Individual dimensions (voice & accountability, rule of law, etc.) may have heterogeneous relationships with sustainability that are masked in the composite.

---

### In plain English: Countries with stronger governance consistently achieve higher sustainability scores — the pattern is clear and robust in every cross-sectional test, but the most conservative within-country analysis (Fixed Effects over just four years) falls just short of the 5% significance threshold, meaning we cannot formally rule out chance for within-country year-to-year governance changes in this short window.
"""
with open(OUT / f"{prefix}_synthesis.md", "w") as f:
    f.write(synthesis_text)
print(f"✅ Saved: outputs/{prefix}_synthesis.md")
print(f"   Word count: {len(synthesis_text.split())} words")

✅ Saved: outputs/h2_synthesis.md
   Word count: 731 words
